# Terrain Graph Node Counts Analysis

This notebook analyzes the txt terrain graph datasets and reports the number of nodes in each full-size terrain graph.

In [ ]:
import os
import glob
import pandas as pd
from pathlib import Path

## Read Terrain Graph Files

Each .txt file contains:
- First line: dimensions (rows cols)
- Remaining lines: height values for the terrain grid

The total number of nodes = rows × cols

In [ ]:
def get_terrain_info(filepath):
    """
    Read a terrain graph txt file and extract dimensions.
    
    Parameters:
    -----------
    filepath : str
        Path to the .txt terrain file
    
    Returns:
    --------
    dict : Dictionary with filename, rows, cols, and total nodes
    """
    with open(filepath, 'r') as f:
        first_line = f.readline().strip()
        rows, cols = map(int, first_line.split())
    
    filename = os.path.basename(filepath)
    total_nodes = rows * cols
    
    return {
        'filename': filename,
        'rows': rows,
        'cols': cols,
        'total_nodes': total_nodes,
        'dimensions': f"{rows}x{cols}"
    }

In [ ]:
# Path to data directory
data_dir = Path('../data')

# Find all .txt files in the data directory
txt_files = sorted(data_dir.glob('*.txt'))

print(f"Found {len(txt_files)} terrain graph files:")
for f in txt_files:
    print(f"  - {f.name}")

## Analyze Node Counts

In [ ]:
# Collect information from all terrain files
terrain_info = []

for txt_file in txt_files:
    info = get_terrain_info(txt_file)
    terrain_info.append(info)
    print(f"Processing: {info['filename']}")

# Create DataFrame for better visualization
df = pd.DataFrame(terrain_info)
print("\nCompleted processing all files.")

## Summary Table

In [ ]:
# Display the results
print("\n" + "="*70)
print("TERRAIN GRAPH NODE COUNTS")
print("="*70 + "\n")

df_display = df[['filename', 'dimensions', 'total_nodes']].copy()
df_display['total_nodes'] = df_display['total_nodes'].apply(lambda x: f"{x:,}")

print(df_display.to_string(index=False))
print("\n" + "="*70)

In [ ]:
# More detailed DataFrame view with styling
df

## Detailed Statistics

In [ ]:
print("\nDetailed Statistics:")
print(f"  Total terrain graphs: {len(df)}")
print(f"  Average nodes per graph: {df['total_nodes'].mean():,.0f}")
print(f"  Median nodes per graph: {df['total_nodes'].median():,.0f}")
print(f"  Min nodes: {df['total_nodes'].min():,} ({df.loc[df['total_nodes'].idxmin(), 'filename']})")
print(f"  Max nodes: {df['total_nodes'].max():,} ({df.loc[df['total_nodes'].idxmax(), 'filename']})")
print(f"  Total nodes across all graphs: {df['total_nodes'].sum():,}")

## Visualization

In [ ]:
import matplotlib.pyplot as plt

# Create bar plot of node counts
fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(range(len(df)), df['total_nodes'])
ax.set_xticks(range(len(df)))
ax.set_xticklabels(df['filename'], rotation=45, ha='right')
ax.set_ylabel('Number of Nodes', fontsize=12)
ax.set_xlabel('Terrain Graph File', fontsize=12)
ax.set_title('Node Counts in Full-Size Terrain Graphs', fontsize=14, fontweight='bold')

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, df['total_nodes'])):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:,}',
            ha='center', va='bottom', fontsize=9)

# Add grid for better readability
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Create a table visualization with dimensions
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('tight')
ax.axis('off')

table_data = []
for _, row in df.iterrows():
    table_data.append([
        row['filename'],
        row['dimensions'],
        f"{row['total_nodes']:,}"
    ])

table = ax.table(cellText=table_data,
                colLabels=['Filename', 'Dimensions', 'Total Nodes'],
                cellLoc='left',
                loc='center',
                colWidths=[0.4, 0.2, 0.25])

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Style header
for i in range(3):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(table_data) + 1):
    for j in range(3):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#f0f0f0')

plt.title('Terrain Graph Summary Table', fontsize=14, fontweight='bold', pad=20)
plt.show()